Let's start by loading necessary libraries.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import jax 
from jax import numpy as jnp
from flows.types import *
import optax
from flows.utils import *
from flows.models.linear import Linear
from flows.utils import *
import matplotlib.pyplot as plt
from functools import partial
from flows.Bases import Hermite 
from flows.bases import *
import flax.linen as nn
import sys 
import math
import pickle 
from sklearn.linear_model import LinearRegression
from jax import config
from target_functions import *
config.update("jax_debug_nans", True)
plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'Computer Modern Roman'
fontsize_label = 18
fontsize_title = fontsize_label + 1
fontsize_legend = fontsize_label - 4
# size of the ticks 
plt.rcParams['xtick.labelsize'] = fontsize_label - 5
plt.rcParams['ytick.labelsize'] = fontsize_label - 5

print(f"available devices: {jax.devices()}")

available devices: [CpuDevice(id=0)]


## Optimization for a target function decaying algebraically

In [2]:
f = f_algebraic

x = jnp.linspace(-30.999,30.999,20000)

w = np.zeros_like(x)
dx = np.diff(x)
w[1:-1] = (dx[:-1] + dx[1:])/2
w[0] = dx[0]/2
w[-1] = dx[-1]/2
x = x[:,None]

f_ev = f(x, w)
integral = jnp.einsum('j,j->', f_ev, w)
print(f_ev[-1], integral)

2.0727462290778044e-24 0.9999999999999938


In [3]:
def Gaussian(x):
    return (jnp.exp(-x[:,0]**2)/jnp.sqrt(jnp.pi))[:, None]

integral_Gaussian = jnp.einsum('j,j->', Gaussian(x)[:,0], w)
print(integral_Gaussian)

0.9999999999999979


Define a sublinear model and test invertibility

In [ ]:
class SubLin(nn.Module):
    dim: int
    beta_init: float = -1/4
    eps_init: float = 10.

    def setup(self):
        #initialize a linear scalar parameter with a default value of 1. and
        #without bias 
        self.eps = self.param(
            'scalar_weight',
            lambda key: jnp.array(self.eps_init)
        )
        self.beta = self.param(
            'scalar_power',
            lambda key: jnp.array(self.beta_init)
        )
  
    
    @nn.compact
    def __call__(self, x, mode=evaluationMode.direct):
        x = jnp.array(x)
        if mode == evaluationMode.inverse:
            
            y = x  # Here, x holds the transformed value
            
            def dF(y):
                return 1/((y**2+self.eps)**self.beta) - 2*y**2*self.beta/((y**2+self.eps)**(self.beta+1))

            def F(y):
                return y/((y**2 + self.eps)**self.beta) - x
            
            for k in range(100):
                #print(f"k:{k}, y:{y}")
                y = y - F(y)/dF(y)
             
            return y
            
        else:
            x = x/((x**2 + self.eps)**self.beta)
            
            return x

class SubLin_fixedalpha(nn.Module):
    dim: int
    beta_init: float = -1/4
    eps_init: float = 8.
  
    def setup(self):
        #initialize a linear scalar parameter with a default value of 1. and
        #without bias 
        self.beta = self.param(
            'scalar_power',
            lambda key: jnp.array(self.beta_init)
        )

    
    @nn.compact
    def __call__(self, x, mode=evaluationMode.direct):
        x = jnp.array(x)
        if mode == evaluationMode.inverse:
            
            pass
        else:
            x = x/((x**2 + self.eps_init)**self.beta)
            
            return x

model = SubLin(1, beta_init=1/4)
#model = SubLin_fixedalpha(1, beta_init=1/4, eps_init=1.)

In [ ]:
params = model.init(jax.random.PRNGKey(0), x, mode=evaluationMode.direct)
#y_forward = model.apply(params, x, mode=evaluationMode.inverse)
#full_f = full_f_algebraic

@jax.jit
def l2_error(params, x, w):    
    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = f(x,w)
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    approx_func = jnp.einsum('j,j->j', Gaussian(y)[:,0], det)
    error_values = (true_func - approx_func)**2
    loss = jnp.sqrt(jnp.einsum('i,i->', error_values, w))
    return loss

# another loss to punish more deviations in the tail 
@jax.jit
def error_tail(params, x, w):    
    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = f(x,w)
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    approx_func = jnp.einsum('j,j->j', Gaussian(y)[:,0], det)
    error_values = (true_func - approx_func)**2
    loss = jnp.sqrt(jnp.einsum('i,i->', error_values*(1+x[:,0]**2 / (true_func + 1e-30)), w))
    return loss

#0.9
def total_loss(params, x, w, alpha=0.9):
    return alpha*l2_error(params, x, w) + (1-alpha)*error_tail(params, x, w)


nograd = lambda x: jax.lax.stop_gradient(x)
loss_grad_fn = jax.value_and_grad(total_loss)#l2_error)
loss_fn = total_loss#l2_error 

@jax.jit
def update_params(carry, args):
    params, opt_state, loss = carry
    loss_val, grad = loss_grad_fn(params, *args)
    updates, opt_state = optimizer.update(grad, opt_state)
    params = optax.apply_updates(params, updates)

    return (params, opt_state, loss+loss_val), 0

learning_rate = 1e-3
n_epochs = 30000
nmax = 0


schedule = optax.cosine_decay_schedule(learning_rate, n_epochs, alpha=5e-5/learning_rate)
optimizer = optax.chain(
    #optax.clip_by_global_norm(1.0),
    optax.adam(schedule),
)
#optimizer = optax.adam(learning_rate=learning_rate)
opt_state = optimizer.init(params)
losses = []
for epoch in range(1,n_epochs):
    carry = (params, opt_state, 0)
    carry, _ = update_params(carry, (nograd(x), nograd(w)))
    params, opt_state, loss = carry 
    losses.append(loss)
    if epoch % 100 == 0:
        l2_loss = l2_error(params, x, w)
        loss_tail = error_tail(params, x, w)
        print(f"Epoch: {epoch}, Loss: {loss}, L2 Loss: {l2_loss}, Tail Loss: {loss_tail}")
        losses.append(loss)

Epoch: 100, Loss: 221.32595245984913, L2 Loss: 0.7765630170169142, Tail Loss: 2186.7397151623345
Epoch: 200, Loss: 115.71696962064364, L2 Loss: 0.7663960598014825, Tail Loss: 1144.6189364811016
Epoch: 300, Loss: 76.93741635568648, L2 Loss: 0.7590221147512085, Tail Loss: 759.907587997218
Epoch: 400, Loss: 56.92175483234597, L2 Loss: 0.75302830943273, Tail Loss: 560.9302768747233
Epoch: 500, Loss: 44.768666744896386, L2 Loss: 0.747882291702223, Tail Loss: 439.982664993181
Epoch: 600, Loss: 36.63722527363165, L2 Loss: 0.7433164353845533, Tail Loss: 359.00575326157997
Epoch: 700, Loss: 30.83159631929391, L2 Loss: 0.7391748763209084, Tail Loss: 301.16703321480156
Epoch: 800, Loss: 26.48880932928563, L2 Loss: 0.735357959160842, Tail Loss: 257.89106812193273
Epoch: 900, Loss: 23.12418793703764, L2 Loss: 0.7317976706306689, Tail Loss: 224.3576508823115
Epoch: 1000, Loss: 20.444936933887632, L2 Loss: 0.7284452881597632, Tail Loss: 197.65309508914302
Epoch: 1100, Loss: 18.263950627461906, L2 Los

In [6]:
print(params)
sing_loc = 1/((params['params']['scalar_weight']-1)**params['params']['scalar_power'])
print(f"singularity location: {sing_loc}")

{'params': {'scalar_power': Array(-0.38783519, dtype=float64), 'scalar_weight': Array(12.53109693, dtype=float64)}}
singularity location: 2.5812543071686735


In [7]:
filename = "simulations_data/sublin_algebraic.pkl"
with open(filename, "wb") as f:
    pickle.dump(params, f)

filename_save = f"simulations_data/sublin_algebraic.npz"
np.savez(filename_save, epochs=n_epochs, losses_nl=losses, sing_loc=sing_loc)

In [19]:
f = f_super_Gaussian

x = jnp.linspace(-12.999,12.999,10000)

w = np.zeros_like(x)
dx = np.diff(x)
w[1:-1] = (dx[:-1] + dx[1:])/2
w[0] = dx[0]/2
w[-1] = dx[-1]/2
x = x[:,None]

f_ev = f(x, w)
integral = jnp.einsum('j,j->', f_ev, w)
print(f_ev[-1], integral)


0.0 1.0


## Optimization for a model decaying as a super Gaussian

In [ ]:
#model = SubLin(1)
eps_init = 2.45
model = SubLin_fixedalpha(1, beta_init=-0.45, eps_init=eps_init)
params = model.init(jax.random.PRNGKey(0), x, mode=evaluationMode.direct)

@jax.jit
def l2_error(params, x, w):    
    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = f(x,w)
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    approx_func = jnp.einsum('j,j->j', Gaussian(y)[:,0], det)
    error_values = (true_func - approx_func)**2
    loss = jnp.sqrt(jnp.einsum('i,i->', error_values, w))
    return loss

# another loss to punish more deviations in the tail 
@jax.jit
def error_tail(params, x, w):    
    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = f(x,w)
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    approx_func = jnp.einsum('j,j->j', Gaussian(y)[:,0], det)
    error_values = (true_func - approx_func)**2
    loss = jnp.sqrt(jnp.einsum('i,i->', error_values*(1+x[:,0]**2 / (true_func + 1e-30)), w))
    return loss

#0.3
def total_loss(params, x, w, alpha=0.3):
    return alpha*l2_error(params, x, w) + (1-alpha)*error_tail(params, x, w)

    #return error_log_ratio(params, x, w)

nograd = lambda x: jax.lax.stop_gradient(x)
loss_grad_fn = jax.value_and_grad(total_loss) #l2_error)
loss_fn = total_loss #l2_error 

@jax.jit
def update_params(carry, args):
    params, opt_state, loss = carry
    loss_val, grad = loss_grad_fn(params, *args)
    updates, opt_state = optimizer.update(grad, opt_state)
    params = optax.apply_updates(params, updates)

    return (params, opt_state, loss+loss_val), 0


learning_rate = 3e-6
n_epochs = 10000

schedule = optax.cosine_decay_schedule(learning_rate, n_epochs, alpha=5e-5/learning_rate)
optimizer = optax.chain(
    optax.adam(schedule),
)
opt_state = optimizer.init(params)
losses = []
for epoch in range(1,n_epochs):
    carry = (params, opt_state, 0)
    carry, _ = update_params(carry, (nograd(x), nograd(w)))
    params, opt_state, loss = carry 
    losses.append(loss)
    
    if epoch % 100 == 0 or epoch == 1:
        l2_loss = l2_error(params, x, w)
        loss_tail = error_tail(params, x, w)
        print(f"Epoch: {epoch}, Loss: {loss}, L2 Loss: {l2_loss}, Tail Loss: {loss_tail}")
        losses.append(loss)

Epoch: 1, Loss: 0.6419871874779729, L2 Loss: 0.4615700738912756, Tail Loss: 0.7193055226984704
Epoch: 100, Loss: 0.6417592068289496, L2 Loss: 0.46137715692381653, Tail Loss: 0.719062500639509
Epoch: 200, Loss: 0.6415270504728684, L2 Loss: 0.46118075730551406, Tail Loss: 0.7188149803050488
Epoch: 300, Loss: 0.6412912363069202, L2 Loss: 0.4609813165895486, Tail Loss: 0.718563512415232
Epoch: 400, Loss: 0.6410499834919203, L2 Loss: 0.46077733295188905, Tail Loss: 0.7183061965562184
Epoch: 500, Loss: 0.6408015122044558, L2 Loss: 0.46056730765583326, Tail Loss: 0.7180411324646738
Epoch: 600, Loss: 0.6405440451773932, L2 Loss: 0.4603497464938397, Tail Loss: 0.717766421609458
Epoch: 700, Loss: 0.6402758091605243, L2 Loss: 0.4601231611983964, Tail Loss: 0.7174801686724591
Epoch: 800, Loss: 0.6399950363023114, L2 Loss: 0.45988607082257393, Tail Loss: 0.7171804829297388
Epoch: 900, Loss: 0.6396999654542876, L2 Loss: 0.45963700309125355, Tail Loss: 0.7168654795347681
Epoch: 1000, Loss: 0.63938884

In [21]:
print(params)
filename = "simulations_data/sublin_superGaussian.pkl"
with open(filename, "wb") as f:
    pickle.dump(params, f)

filename_save = f"simulations_data/sublin_superGaussian.npz"
np.savez(filename_save, epochs=n_epochs, losses_nl=losses,)

{'params': {'scalar_power': Array(-0.30185468, dtype=float64)}}


## Approximation using perturbed Hermite

In [ ]:


def compute_coeff_uj(params, decay):
    
    if decay == "algebraic":
        x = jnp.linspace(-30.999, 30.999, 10000)
        full_f = full_f_algebraic
    elif decay == "superGaussian":
        x = jnp.linspace(-12.999, 12.999, 10000)
        full_f = full_f_super_Gaussian
    
    w = np.zeros_like(x)
    dx = np.diff(x)
    w[1:-1] = (dx[:-1] + dx[1:])/2
    w[0] = dx[0]/2
    w[-1] = dx[-1]/2
    x = x[:,None]

    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = full_f(x,w)
    #det = abs_det_jac_x(model, params, r, mode=evaluationMode.direct)
    psi_ev = psi_o(y)
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    det = jnp.sqrt(det)
    weight = jnp.exp(-y[:,0]**2/2)  
    psi_ev = jnp.einsum('jk,j,j->jk', psi_ev, weight, det)
    approx_coeffs = jnp.einsum('j,jk,j->k', true_func, psi_ev, w)
    return approx_coeffs
   
def l2_error_uj(params, decay):    
    #det = abs_det_jac_x(model, params, r, mode=evaluationMode.direct)
    if decay == "algebraic":
        x = jnp.linspace(-30.999, 30.999, 10000)
        full_f = full_f_algebraic
    elif decay == "superGaussian":
        x = jnp.linspace(-12.999, 12.999, 10000)
        full_f = full_f_super_Gaussian
    
    w = np.zeros_like(x)
    dx = np.diff(x)
    w[1:-1] = (dx[:-1] + dx[1:])/2
    w[0] = dx[0]/2
    w[-1] = dx[-1]/2
    x = x[:,None]

    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = full_f(x,w)
    psi_ev = psi_o(y) #pr?
    weight = jnp.exp(-y[:,0]**2/2)  
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    det = jnp.sqrt(det)

    approx_coeffs = compute_coeff_uj(params, decay)
    approx_func = jnp.einsum('k,jk,j,j->j', approx_coeffs, psi_ev, det, weight)

    error_values = (true_func - approx_func)**2
    #loss = jnp.mean(error_values)
    loss = jnp.sqrt(jnp.einsum('i,i->', error_values, w))
    return loss


def linf_error_uj(params, decay):  
    if decay == "algebraic":
        x = jnp.linspace(-30.999, 30.999, 10000)
        full_f = full_f_algebraic
    elif decay == "superGaussian":
        x = jnp.linspace(-12.999, 12.999, 10000)
        full_f = full_f_super_Gaussian
    
    w = np.zeros_like(x)
    dx = np.diff(x)
    w[1:-1] = (dx[:-1] + dx[1:])/2
    w[0] = dx[0]/2
    w[-1] = dx[-1]/2
    x = x[:,None]

    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = full_f(x,w)
    psi_ev = psi_o(y) #pr?
    weight = jnp.exp(-y[:,0]**2/2)  
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    det = jnp.sqrt(det)

    approx_coeffs = compute_coeff_uj(params, decay)
    approx_func = jnp.einsum('k,jk,j,j->j', approx_coeffs, psi_ev, det, weight)

    error_values = jnp.abs(true_func - approx_func)
    loss = jnp.amax(error_values)
    return loss

n_values = np.arange(2,48,4)
list_decay = ["algebraic", "superGaussian"]
list_filename = [f"simulations_data/sublin_{decay}.pkl" for decay in list_decay]
#n_layers = 2
#list_decay = ["algebraic"]
#list_filename = ["simulations_data/iResNet_algebraic_scaled_erf_nlayers_5.pkl"]


for decay, filename in zip(list_decay, list_filename):
    losses_nl = []
    lossesinf_nl = []
    size_basis = []
    with open(filename, "rb") as f:
        params = pickle.load(f)
    print(f"decay: {decay}, parameters: {params}")
    if decay == "algebraic":
        model = SubLin(1,)
    else:
        model = SubLin_fixedalpha(1, beta_init=-0.45, eps_init=eps_init)
        
    for n in n_values:
        nmax = n
        n_basis = [nmax for _ in range(x.shape[1])]
        w_basis = [1 for _ in range(x.shape[1])]
        # Note that when we project we need to multiply by the weights of the basis
      
        basis_r = Hermite.init_basis(n_basis, w_basis, nmax, orthotype=orthoType.ortho) # 
        psi_o = partial(Hermite.batch_basis_values, basis_r)  

        loss = l2_error_uj(params, decay)
        loss_inf = linf_error_uj(params, decay)
        print(f"n: {n}, L2 error: {loss}, Linf error: {loss_inf}")
        losses_nl.append(loss)
        lossesinf_nl.append(loss_inf)
        size_basis.append(n+1)
        #plot_uj(params, x, w)
    np.savez(f"simulations_data/losses_SubLin_{decay}.npz", size_basis=size_basis, losses_nl=losses_nl, lossesinf_nl=lossesinf_nl)

decay: algebraic, parameters: {'params': {'scalar_power': Array(-0.38783519, dtype=float64), 'scalar_weight': Array(12.53109693, dtype=float64)}}
n: 2, L2 error: 0.030223061304359523, Linf error: 0.036055899516621
n: 6, L2 error: 0.0042661253060829255, Linf error: 0.005020579665562597
n: 10, L2 error: 0.0015505392661385817, Linf error: 0.0018977792745963583
n: 14, L2 error: 0.0008891857164533629, Linf error: 0.001194737245274078
n: 18, L2 error: 0.0005804262334758634, Linf error: 0.0007758340711075464
n: 22, L2 error: 0.000401815880164936, Linf error: 0.0005250284254940012
n: 26, L2 error: 0.0002893049673541739, Linf error: 0.00036843844048967524
n: 30, L2 error: 0.00021458355599676584, Linf error: 0.00026637062735139315
n: 34, L2 error: 0.00016296125002150276, Linf error: 0.0001972633451843868
n: 38, L2 error: 0.00012615904183464817, Linf error: 0.00014900470755712481
n: 42, L2 error: 9.923739914416746e-05, Linf error: 0.00011438015016884422
n: 46, L2 error: 7.91147852145048e-05, Linf

In [11]:


def compute_coeff_uj(params, decay):
    
    if decay == "algebraic":
        x = jnp.linspace(-30.999, 30.999, 10000)
        full_f = full_f_algebraic
    elif decay == "superGaussian":
        x = jnp.linspace(-12.999, 12.999, 10000)
        full_f = full_f_super_Gaussian
    
    w = np.zeros_like(x)
    dx = np.diff(x)
    w[1:-1] = (dx[:-1] + dx[1:])/2
    w[0] = dx[0]/2
    w[-1] = dx[-1]/2
    x = x[:,None]

    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = full_f(x,w)
    #det = abs_det_jac_x(model, params, r, mode=evaluationMode.direct)
    psi_ev = psi_o(y)
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    det = jnp.sqrt(det)
    weight = jnp.exp(-y[:,0]**2/2)  
    psi_ev = jnp.einsum('jk,j,j->jk', psi_ev, weight, det)
    approx_coeffs = jnp.einsum('j,jk,j->k', true_func, psi_ev, w)
    return approx_coeffs
   
def l2_error_uj(params, decay):    
    #det = abs_det_jac_x(model, params, r, mode=evaluationMode.direct)
    if decay == "algebraic":
        x = jnp.linspace(-30.999, 30.999, 10000)
        full_f = full_f_algebraic
    elif decay == "superGaussian":
        x = jnp.linspace(-12.999, 12.999, 10000)
        full_f = full_f_super_Gaussian
    
    w = np.zeros_like(x)
    dx = np.diff(x)
    w[1:-1] = (dx[:-1] + dx[1:])/2
    w[0] = dx[0]/2
    w[-1] = dx[-1]/2
    x = x[:,None]

    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = full_f(x,w)
    psi_ev = psi_o(y) #pr?
    weight = jnp.exp(-y[:,0]**2/2)  
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    det = jnp.sqrt(det)

    approx_coeffs = compute_coeff_uj(params, decay)
    approx_func = jnp.einsum('k,jk,j,j->j', approx_coeffs, psi_ev, det, weight)

    error_values = (true_func - approx_func)**2
    #loss = jnp.mean(error_values)
    loss = jnp.sqrt(jnp.einsum('i,i->', error_values, w))
    return loss


def linf_error_uj(params, decay):  
    if decay == "algebraic":
        x = jnp.linspace(-30.999, 30.999, 10000)
        full_f = full_f_algebraic
    elif decay == "superGaussian":
        x = jnp.linspace(-12.999, 12.999, 10000)
        full_f = full_f_super_Gaussian
    
    w = np.zeros_like(x)
    dx = np.diff(x)
    w[1:-1] = (dx[:-1] + dx[1:])/2
    w[0] = dx[0]/2
    w[-1] = dx[-1]/2
    x = x[:,None]

    y = model.apply(params, x, mode=evaluationMode.direct)
    true_func = full_f(x,w)
    psi_ev = psi_o(y) #pr?
    weight = jnp.exp(-y[:,0]**2/2)  
    det = abs_det_jac_x(model, params, x, mode=evaluationMode.direct)
    det = jnp.sqrt(det)

    approx_coeffs = compute_coeff_uj(params, decay)
    approx_func = jnp.einsum('k,jk,j,j->j', approx_coeffs, psi_ev, det, weight)

    error_values = jnp.abs(true_func - approx_func)
    loss = jnp.amax(error_values)
    return loss

n_values = np.arange(2,48,4)
list_decay = ["superGaussian"]


for decay in list_decay:
    losses_nl = []
    lossesinf_nl = []
    size_basis = []
    model = SubLin_fixedalpha(1, beta_init=-0.45, eps_init=8)
    
    params = model.init(jax.random.PRNGKey(0), x, mode=evaluationMode.direct)
    
    print(f"total loss of the model at n=0", total_loss(params, x, w,))
    for n in n_values:
        nmax = n
        n_basis = [nmax for _ in range(x.shape[1])]
        w_basis = [1 for _ in range(x.shape[1])]
        # Note that when we project we need to multiply by the weights of the basis
      
        basis_r = Hermite.init_basis(n_basis, w_basis, nmax, orthotype=orthoType.ortho) # 
        psi_o = partial(Hermite.batch_basis_values, basis_r)  

        loss = l2_error_uj(params, decay)
        loss_inf = linf_error_uj(params, decay)
        print(f"n: {n}, L2 error: {loss}, Linf error: {loss_inf}")
        losses_nl.append(loss)
        lossesinf_nl.append(loss_inf)
        size_basis.append(n+1)
        #plot_uj(params, x, w)
    np.savez(f"simulations_data/losses_SubLin_{decay}_manual.npz", size_basis=size_basis, losses_nl=losses_nl, lossesinf_nl=lossesinf_nl)

total loss of the model at n=0 0.9775083571182825
n: 2, L2 error: 0.24054612482628812, Linf error: 0.3353583346802134
n: 6, L2 error: 0.06073189255614067, Linf error: 0.0700664302823854
n: 10, L2 error: 0.010102436158974078, Linf error: 0.014708674919439994
n: 14, L2 error: 0.0007145951631501759, Linf error: 0.001058724912671673
n: 18, L2 error: 9.004203905371824e-06, Linf error: 9.414073638650843e-06
n: 22, L2 error: 4.0151711494914623e-07, Linf error: 5.427564267929163e-07
n: 26, L2 error: 1.0051936492108254e-08, Linf error: 1.4704434657226241e-08
n: 30, L2 error: 2.3995463121571317e-10, Linf error: 3.608365345379817e-10
n: 34, L2 error: 6.41255691878791e-12, Linf error: 9.690295367639974e-12
n: 38, L2 error: 1.9709462661361217e-13, Linf error: 2.9539886429991056e-13
n: 42, L2 error: 6.6284217548055555e-15, Linf error: 9.619361773387817e-15
n: 46, L2 error: 6.0864753691152e-16, Linf error: 8.881784197001252e-16


In [16]:
print(model.eps_init)

8
